In [35]:
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import operator
import re
load_dotenv()

True

In [36]:
class Task(BaseModel):
    id: int
    title: str
    brief: str = Field(...,description="What to cover")

In [37]:
class Plan(BaseModel):
    blog_title: str
    tasks: list[Task]

In [38]:
class Section(TypedDict):
    id: int
    content: str

In [39]:
class State(TypedDict):
    topic: str
    plan: Plan
    sections: Annotated[list[Section], operator.add]
    final: str

In [40]:
llm = ChatOpenAI(model="gpt-4.1-mini")

In [41]:
def orchestractor(state: State):
    plan = llm.with_structured_output(Plan).invoke(
        [
            SystemMessage(content="Create a blog plan with 5-7 sections on the following topic. Assign sequential task IDs starting from 1."),
            HumanMessage(content=f"Topic: {state['topic']}")
        ]
    )
    return {"plan":plan}

In [42]:
def fanout(state: State):
    return [Send("worker",{"task":task, "topic":state["topic"], "plan":state["plan"]}) for task in state["plan"].tasks]

In [43]:
def worker(payload: dict)->dict:
    task = payload["task"]
    topic = payload["topic"]
    plan = payload["plan"]

    blog_title = plan.blog_title

    section_md = llm.invoke([
        SystemMessage(content="write one clean Markdown section."),
        HumanMessage(content=f"Blog:{blog_title}\n"f"Topic:{topic}\n\n"f"Section:{task.title}\n"f"Brief:{task.brief}\n\n""Return only the section content in Markdown")
    ]
    ).content.strip()

    return {"sections":[{"id":task.id,"content":section_md}]}

In [48]:
from pathlib import Path

def reducer(state: State):
    title = state['plan'].blog_title
    sections = sorted(state["sections"],key=lambda section: section["id"])
    body = "\n\n".join(section["content"] for section in sections).strip()

    final_md = f"# {title}\n\n{body}\n"

    filename = title.lower().strip().replace(" ", "_") + ".md"
    output_path = Path(filename)
    output_path.write_text(final_md,encoding="utf-8")
    return {"final": final_md}

In [49]:
g = StateGraph(State)
g.add_node("orchestrator",orchestractor)
g.add_node("worker",worker)
g.add_node("reducer",reducer)

In [50]:
g.add_edge(START,"orchestrator")
g.add_conditional_edges("orchestrator", fanout, ["worker"])
g.add_edge("worker", "reducer")
g.add_edge("reducer",END)

app = g.compile()


In [51]:
out = app.invoke({"topic":"Film Making"})

In [ ]:
llm.invoke("who is pawan kalyan")